In [21]:
# %%
"""
📌 Notebook: 03_feature_engineering.ipynb

🎯 Purpose:
Extract meaningful features from raw sensor data to improve model learning.

Features created:
- Acceleration magnitude
- Gyroscope magnitude
- Jerk (absolute rate of change of acceleration)
- Rolling statistics (smoothness)
- Signal variability (driving stability)

🔥 Improvements:
- Absolute jerk for better behavior modeling
- Proper NaN handling in rolling features
- Timestamp removed (not useful for ML)
"""

import pandas as pd
import numpy as np

print("✅ Libraries loaded.")

✅ Libraries loaded.


In [22]:
# %%
# Load dataset
input_path = "../processed_data/cleaned_dataset.csv"
dataset = pd.read_csv(input_path)

print(f"✅ Dataset loaded. Shape: {dataset.shape}")

✅ Dataset loaded. Shape: (2750848, 9)


In [23]:
# %%
"""
🔹 Step 1: Magnitude Features

These represent overall motion intensity and rotation intensity.
"""

# Acceleration magnitude
dataset["acc_mag"] = np.sqrt(
    dataset["X_Acc"]**2 + 
    dataset["Y_Acc"]**2 + 
    dataset["Z_Acc"]**2
)

# Gyroscope magnitude
dataset["gyro_mag"] = np.sqrt(
    dataset["X_Gyro"]**2 + 
    dataset["Y_Gyro"]**2 + 
    dataset["Z_Gyro"]**2
)

print("✅ Magnitude features created.")

✅ Magnitude features created.


In [24]:
# %%
"""
🔹 Step 2: Jerk (IMPORTANT)

Jerk = rate of change of acceleration
We use ABSOLUTE jerk to capture intensity of sudden motion.
"""

dataset["jerk"] = dataset.groupby("trip_id")["acc_mag"].diff().abs().fillna(0)

print("✅ Absolute jerk feature created.")

✅ Absolute jerk feature created.


In [25]:
# %%
"""
🔹 Step 3: Rolling Statistics (FINAL + OPTIMIZED 🔥)

🎯 Purpose:
Capture driving smoothness and local behavior patterns.

- Smooth driving → low rolling std
- Aggressive driving → high rolling std

✅ Improvements:
- No chained assignment warnings
- No deprecated fillna(method=...)
- Proper handling per trip_id (no data leakage)
- Robust NaN handling
"""

window_size = 5  # you can tune later

# -------------------------------
# 🚗 Acceleration Rolling Features
# -------------------------------
dataset["acc_roll_mean"] = (
    dataset.groupby("trip_id")["acc_mag"]
    .rolling(window_size)
    .mean()
    .reset_index(level=0, drop=True)
)

dataset["acc_roll_std"] = (
    dataset.groupby("trip_id")["acc_mag"]
    .rolling(window_size)
    .std()
    .reset_index(level=0, drop=True)
)

# -------------------------------
# 🔄 Gyroscope Rolling Features
# -------------------------------
dataset["gyro_roll_mean"] = (
    dataset.groupby("trip_id")["gyro_mag"]
    .rolling(window_size)
    .mean()
    .reset_index(level=0, drop=True)
)

dataset["gyro_roll_std"] = (
    dataset.groupby("trip_id")["gyro_mag"]
    .rolling(window_size)
    .std()
    .reset_index(level=0, drop=True)
)

# -------------------------------
# ⚠️ Handle NaN Values (IMPORTANT)
# -------------------------------

# Backward fill within each trip (no leakage)
dataset["acc_roll_mean"] = dataset.groupby("trip_id")["acc_roll_mean"].bfill()
dataset["gyro_roll_mean"] = dataset.groupby("trip_id")["gyro_roll_mean"].bfill()

# Fill std NaNs using global mean (safe fallback)
dataset["acc_roll_std"] = dataset["acc_roll_std"].fillna(dataset["acc_roll_std"].mean())
dataset["gyro_roll_std"] = dataset["gyro_roll_std"].fillna(dataset["gyro_roll_std"].mean())

print("✅ Rolling features created (clean + optimized).")

✅ Rolling features created (clean + optimized).


In [26]:
# %%
"""
🔹 Step 4: Stability / Variability Features

These capture how consistent or unstable driving is.
"""

dataset["acc_var"] = dataset.groupby("trip_id")["acc_mag"].transform("var")
dataset["gyro_var"] = dataset.groupby("trip_id")["gyro_mag"].transform("var")

print("✅ Variability features created.")

✅ Variability features created.


In [27]:
# %%
"""
🔹 Step 5: Remove unwanted columns

Timestamp is not useful for ML models.
"""

dataset = dataset.drop(columns=["Timestamp"], errors="ignore")

print("✅ Timestamp removed.")

✅ Timestamp removed.


In [28]:
# %%
"""
🔹 Step 6: Final Check
"""

print("📊 Final columns:")
print(dataset.columns.tolist())

display(dataset.head())

📊 Final columns:
['X_Acc', 'Y_Acc', 'Z_Acc', 'X_Gyro', 'Y_Gyro', 'Z_Gyro', 'rating', 'trip_id', 'acc_mag', 'gyro_mag', 'jerk', 'acc_roll_mean', 'acc_roll_std', 'gyro_roll_mean', 'gyro_roll_std', 'acc_var', 'gyro_var']


,X_Acc,Y_Acc,Z_Acc,X_Gyro,Y_Gyro,Z_Gyro,rating,trip_id,acc_mag,gyro_mag,jerk,acc_roll_mean,acc_roll_std,gyro_roll_mean,gyro_roll_std,acc_var,gyro_var
0,-0.79173,-4.23709,9.16870,-4.71483,6.08045,12.31348,1,1,10.131378,14.519755,0.000000,9.781454,0.939044,17.201849,6.059045,0.659909,108.263774
1,-0.44716,2.18525,10.64028,-21.07940,5.18784,4.70720,1,1,10.871561,22.212891,0.740182,9.781454,0.939044,17.201849,6.059045,0.659909,108.263774
2,-1.50957,-2.00219,8.74278,17.42121,1.83863,12.03883,1,1,9.095261,21.255883,1.776300,9.781454,0.939044,17.201849,6.059045,0.659909,108.263774
3,-0.76301,-0.42532,9.33620,1.31222,1.78523,5.42052,1,1,9.376978,5.855852,0.281717,9.781454,0.939044,17.201849,6.059045,0.659909,108.263774
4,-0.18395,-1.60259,9.29313,-21.75076,0.30898,-4.25326,1,1,9.432094,22.164865,0.055116,9.781454,0.718904,17.201849,7.108138,0.659909,108.263774


In [29]:
# %%
"""
💾 Step 7: Save feature dataset
"""

output_path = "../processed_data/feature_dataset.csv"
dataset.to_csv(output_path, index=False)

print(f"✅ Feature dataset saved at: {output_path}")

✅ Feature dataset saved at: ../processed_data/feature_dataset.csv
